# Sentiment Analysis — Amazon Product Reviews

Notebook interativo para explorar e testar o pipeline de análise de sentimentos.

> **Antes de rodar:** execute este notebook a partir da raiz do projeto (`project-sentiment-analysis/`),
> ou ajuste o `sys.path` na célula abaixo.

In [ ]:
import sys
import os

# Garante que imports e caminhos relativos funcionem a partir da raiz do projeto
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("Raiz do projeto:", ROOT)
print("Diretório de trabalho:", os.getcwd())

## 1. Carregamento e inspeção dos dados

In [2]:
from src.data.loader import load_data, inspect_data
from src.utils.config import DATA_PATH

df = load_data(DATA_PATH)
inspect_data(df)
df[["reviews.text", "reviews.rating"]].head()

FileNotFoundError: Dataset not found: data/raw/reviews.csv

## 2. Pré-processamento

In [ ]:
from src.preprocessing.transform import preprocess_dataset
from src.utils.config import TEXT_COLUMN, LABEL_COLUMN

df_clean = preprocess_dataset(df)
print(f"Amostras após pré-processamento: {len(df_clean)}")
print(f"Distribuição de sentimento:\n{df_clean[LABEL_COLUMN].value_counts()}")
df_clean[[TEXT_COLUMN, LABEL_COLUMN]].head()

## 3. Vetorização TF-IDF e normalização NumPy

In [ ]:
import numpy as np
from src.preprocessing.transform import vectorize_texts, normalize_features
from src.training.train import split_dataset
from src.utils.config import TEST_SIZE, MAX_FEATURES

x_train, x_test, y_train, y_test = split_dataset(df_clean, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE)

vectorizer, X_train_np = vectorize_texts(x_train, MAX_FEATURES)
X_test_np = vectorizer.transform(x_test).toarray()

X_train_np = normalize_features(X_train_np)
X_test_np  = normalize_features(X_test_np)

print(f"X_train shape : {X_train_np.shape}")
print(f"X_test  shape : {X_test_np.shape}")
print(f"Norma média (train): {np.linalg.norm(X_train_np, axis=1).mean():.4f}")

## 4. Conversão para tensores PyTorch e DataLoaders

In [ ]:
import torch
from src.training.train import to_tensors, create_dataloader
from src.utils.config import BATCH_SIZE

X_train_t, y_train_t = to_tensors(X_train_np, y_train.to_numpy())
X_test_t,  y_test_t  = to_tensors(X_test_np,  y_test.to_numpy())

train_loader = create_dataloader(X_train_t, y_train_t, BATCH_SIZE)
val_loader   = create_dataloader(X_test_t,  y_test_t,  BATCH_SIZE, shuffle=False)

print(f"X_train tensor : {X_train_t.shape} | dtype: {X_train_t.dtype}")
print(f"y_train tensor : {y_train_t.shape} | dtype: {y_train_t.dtype}")
print(f"Batches por época (train): {len(train_loader)}")

## 5. Modelo PyTorch — arquitetura

In [ ]:
from src.models.model import build_model
from src.utils.config import HIDDEN_DIM, OUTPUT_DIM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

model = build_model(MAX_FEATURES, HIDDEN_DIM, OUTPUT_DIM).to(device)
print(model)

## 6. Treinamento

In [ ]:
import torch.nn as nn
from src.training.train import run_training
from src.utils.config import LEARNING_RATE, NUM_EPOCHS

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

model = run_training(
    model, train_loader, val_loader,
    optimizer, criterion, NUM_EPOCHS, device
)

## 7. Avaliação

In [ ]:
from src.models.model import predict
from src.evaluation.metrics import evaluate_model, print_report

y_pred = predict(model, X_test_t, device)
metrics = evaluate_model(y_test.to_numpy(), y_pred)
print_report(metrics)

## 8. Matriz de confusão

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test.to_numpy(), y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Inferência — teste com textos novos

In [ ]:
from src.inference.predict import predict_batch

exemplos = [
    "This product is absolutely amazing, I love it!",
    "Terrible quality, broke after one day. Total waste of money.",
    "It works as expected, nothing special but does the job.",
    "Best purchase I have ever made, highly recommend!",
    "Very disappointed, does not work at all.",
]

labels = predict_batch(exemplos, model, vectorizer, device)
sentimentos = ["NEGATIVE" if l == 0 else "POSITIVE" for l in labels]

for texto, sentimento in zip(exemplos, sentimentos):
    print(f"[{sentimento}] {texto}")

## 10. Salvar modelo e artefatos

In [ ]:
import json
import pickle
from src.models.model import save_model
from src.utils.config import MODEL_PATH, VECTORIZER_PATH, METRICS_PATH

save_model(model, MODEL_PATH)
with open(VECTORIZER_PATH, "wb") as f:
    pickle.dump(vectorizer, f)
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Modelo salvo em    : {MODEL_PATH}")
print(f"Vectorizer salvo em: {VECTORIZER_PATH}")
print(f"Métricas salvas em : {METRICS_PATH}")